In [2]:
import torch

print("Torch Version", torch.__version__)
print("CUDA Avaialble:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

Torch Version 2.5.1+cu121
CUDA Avaialble: True
GPU name: NVIDIA GeForce RTX 4070 Laptop GPU


In [2]:
from datasets import load_dataset

dataset = load_dataset('wikitext','wikitext-2-raw-v1')

train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

print(train_data[0])

c:\Users\sidsr\AI-Research\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'text': ''}


In [3]:
!pip install -q torch datasets transformers tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import wandb
# safer: store key in environment variable WANDB_API_KEY
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\sidsr\_netrc
wandb: Currently logged in as: siddharthsrinivasan2002 (siddharthsrinivasan2002-national-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import math
import time
from dataclasses import dataclass
from typing import Optional, Dict, Type

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\sidsr\AI-Research\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [4]:
@dataclass
class Config:
    # Data
    dataset_name: str = "wikitext"
    dataset_config: str = "wikitext-2-raw-v1"
    tokenizer_name: str = "gpt2"
    context_len: int = 1024
    batch_size: int = 4
    
    # Model (baseline Transformer)
    vocab_size: int = 50257  # GPT-2 tokenizer vocab size
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 8
    d_ff: int = 2048
    dropout: float = 0.1
    
    # Swappable components
    attention_type: str = "scaled_dot_product"
    positional_encoding_type: str = "learned"
    block_type: str = "standard_transformer_block"
    
    # Training
    epochs: int = 2
    lr: float = 3e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_every_steps: int = 500
    max_eval_batches: int = 100  # cap for faster eval during experimentation
    
cfg = Config()
print(cfg)

Config(dataset_name='wikitext', dataset_config='wikitext-2-raw-v1', tokenizer_name='gpt2', context_len=1024, batch_size=4, vocab_size=50257, d_model=512, n_heads=8, n_layers=8, d_ff=2048, dropout=0.1, attention_type='scaled_dot_product', positional_encoding_type='learned', block_type='standard_transformer_block', epochs=2, lr=0.0003, weight_decay=0.1, grad_clip=1.0, eval_every_steps=500, max_eval_batches=100)


In [5]:
run = wandb.init(
    project="wikitext2-transformer-baseline",
    name=f"baseline_ctx{cfg.context_len}_L{cfg.n_layers}_D{cfg.d_model}_H{cfg.n_heads}",
    config=vars(cfg),
    tags=["baseline", "transformer", "wikitext2", "context-1024"],
    notes="Standard Transformer baseline with modular attention/PE/block."
)

In [6]:
raw_ds = load_dataset(cfg.dataset_name, cfg.dataset_config)
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)

# GPT2 has no pad token by default; for LM training we don't need padding in this setup.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_split(text_list):
    # Concatenate text with EOS separators to get a long stream.
    merged = tokenizer.eos_token.join(text_list)
    ids = tokenizer(merged, return_attention_mask=False, add_special_tokens=False)["input_ids"]
    return torch.tensor(ids, dtype=torch.long)

train_ids = tokenize_split(raw_ds["train"]["text"])
valid_ids = tokenize_split(raw_ds["validation"]["text"])

print("Train tokens:", train_ids.numel())
print("Valid tokens:", valid_ids.numel())
print("Context length:", cfg.context_len)

Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors


Train tokens: 2428601
Valid tokens: 251048
Context length: 1024


In [7]:
class LanguageModelingDataset(Dataset):
    """
    Produces fixed-length chunks:
      x: tokens [t ... t+L-1]
      y: tokens [t+1 ... t+L]
    """
    def __init__(self, token_ids: torch.Tensor, context_len: int):
        self.token_ids = token_ids
        self.context_len = context_len
        self.n_samples = (len(token_ids) - 1) // context_len

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.context_len
        end = start + self.context_len
        x = self.token_ids[start:end]
        y = self.token_ids[start+1:end+1]
        return x, y

train_ds = LanguageModelingDataset(train_ids, cfg.context_len)
valid_ds = LanguageModelingDataset(valid_ids, cfg.context_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

print("Train samples:", len(train_ds))
print("Valid samples:", len(valid_ds))

Train samples: 2371
Valid samples: 245


In [8]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        return x + self.pos_emb(pos)

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)  # (1, max_len, d_model)

    def forward(self, x):
        B, T, D = x.shape
        return x + self.pe[:, :T, :]

POSITIONAL_ENCODING_REGISTRY: Dict[str, Type[nn.Module]] = {
    "learned": LearnedPositionalEmbedding,
    "sinusoidal": SinusoidalPositionalEncoding,
}

In [9]:
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        return x + self.pos_emb(pos)

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)  # (1, max_len, d_model)

    def forward(self, x):
        B, T, D = x.shape
        return x + self.pe[:, :T, :]

POSITIONAL_ENCODING_REGISTRY: Dict[str, Type[nn.Module]] = {
    "learned": LearnedPositionalEmbedding,
    "sinusoidal": SinusoidalPositionalEncoding,
}

In [10]:
class ScaledDotProductCausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv_proj(x)  # (B, T, 3D)
        q, k, v = qkv.chunk(3, dim=-1)

        # (B, H, T, Hd)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, H, T, T)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn_scores = attn_scores.masked_fill(causal_mask, float("-inf"))

        attn = F.softmax(attn_scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)  # (B, H, T, Hd)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.out_proj(out)
        return out

ATTENTION_REGISTRY: Dict[str, Type[nn.Module]] = {
    "scaled_dot_product": ScaledDotProductCausalSelfAttention
}

class StandardTransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float, attention_type: str):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = ATTENTION_REGISTRY[attention_type](d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

BLOCK_REGISTRY: Dict[str, Type[nn.Module]] = {
    "standard_transformer_block": StandardTransformerBlock
}

In [11]:
class TransformerLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        pos_cls = POSITIONAL_ENCODING_REGISTRY[cfg.positional_encoding_type]
        self.pos_enc = pos_cls(cfg.context_len, cfg.d_model)

        block_cls = BLOCK_REGISTRY[cfg.block_type]
        self.blocks = nn.ModuleList([
            block_cls(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout, cfg.attention_type)
            for _ in range(cfg.n_layers)
        ])

        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Optional weight tying (common for language models)
        self.lm_head.weight = self.tok_emb.weight

        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, idx, targets: Optional[torch.Tensor] = None):
        # idx: (B, T)
        x = self.tok_emb(idx)                # (B, T, D)
        x = self.pos_enc(x)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)             # (B, T, V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

model = TransformerLM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")

Model parameters: 51.48M


In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    ppl = math.exp(mean_loss)
    return mean_loss, ppl

def train_one_epoch(model, loader, optimizer, epoch_idx):
    model.train()
    running_loss = 0.0
    total_tokens = 0
    start_time = time.time()

    pbar = tqdm(enumerate(loader), total=len(loader), desc=f"Epoch {epoch_idx+1}")
    for step, (x, y) in pbar:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        running_loss += loss.item()
        total_tokens += x.numel()

        if (step + 1) % 50 == 0:
            avg_loss = running_loss / (step + 1)
            pbar.set_postfix({"train_loss": f"{avg_loss:.4f}"})

        if (step + 1) % cfg.eval_every_steps == 0:
            val_loss, val_ppl = evaluate(model, valid_loader, max_batches=cfg.max_eval_batches)
            print(f"\nStep {step+1}: val_loss={val_loss:.4f}, val_ppl={val_ppl:.2f}")

    elapsed = time.time() - start_time
    train_loss = running_loss / len(loader)
    train_ppl = math.exp(train_loss)
    throughput = total_tokens / elapsed  # tokens/sec
    return train_loss, train_ppl, throughput

In [14]:
wandb.config.update({
    "num_parameters": sum(p.numel() for p in model.parameters()),
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
}, allow_val_change=True)

wandb.watch(model, log="gradients", log_freq=200)  # optional but useful

In [15]:
global_step = epoch_idx * len(loader) + step + 1
wandb.log({
    "train/step_loss": loss.item(),
    "train/lr": optimizer.param_groups[0]["lr"],
}, step=global_step)

NameError: name 'epoch_idx' is not defined

In [17]:
history = []

for epoch in range(cfg.epochs):
    train_loss, train_ppl, tps = train_one_epoch(model, train_loader, optimizer, epoch)
    val_loss, val_ppl = evaluate(model, valid_loader, max_batches=None)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_ppl": train_ppl,
        "val_loss": val_loss,
        "val_ppl": val_ppl,
        "throughput_tokens_per_sec": tps
    }
    history.append(row)
    print(f"\nEpoch {epoch+1} done")
    print(row)

print("\nBaseline complete.")
print("Final validation perplexity:", history[-1]["val_ppl"])
print("Final throughput (tokens/sec):", history[-1]["throughput_tokens_per_sec"])

Epoch 1:  84%|████████▍ | 500/592 [18:49<08:13,  5.36s/it, train_loss=11.3432]


Step 500: val_loss=8.4124, val_ppl=4502.41


Epoch 1: 100%|██████████| 592/592 [20:07<00:00,  2.04s/it, train_loss=11.0674]



Epoch 1 done
{'epoch': 1, 'train_loss': 10.857860830184576, 'train_ppl': 51940.84896456755, 'val_loss': 8.06129539397455, 'val_ppl': 3169.3930890225756, 'throughput_tokens_per_sec': 2007.6882213418958}


Epoch 2:  84%|████████▍ | 500/592 [10:44<06:03,  3.95s/it, train_loss=7.5353]


Step 500: val_loss=7.2635, val_ppl=1427.27


Epoch 2: 100%|██████████| 592/592 [12:04<00:00,  1.22s/it, train_loss=7.4987]



Epoch 2 done
{'epoch': 2, 'train_loss': 7.470141360083142, 'train_ppl': 1754.8547344537935, 'val_loss': 7.161287092393445, 'val_ppl': 1288.568372588858, 'throughput_tokens_per_sec': 3345.04980500344}

Baseline complete.
Final validation perplexity: 1288.568372588858
Final throughput (tokens/sec): 3345.04980500344
